In [32]:
import re
import nltk
import fasttext
from nltk.corpus import stopwords

import numpy as np
import pandas as pd

from keras.models import Model
from keras.utils import pad_sequences
from keras.preprocessing.text import Tokenizer
from keras.layers import Dense, Dropout, Embedding, Input, LSTM

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [3]:
Max_sequence_length = 50
Embedding_dim = 100

In [6]:
df = pd.read_csv('Emotion_Data.csv')

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21459 entries, 0 to 21458
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Text     21459 non-null  object
 1   Emotion  21459 non-null  object
dtypes: object(2)
memory usage: 335.4+ KB


In [8]:
data = df.sample(n=20000)

In [9]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 20000 entries, 7819 to 12312
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Text     20000 non-null  object
 1   Emotion  20000 non-null  object
dtypes: object(2)
memory usage: 468.8+ KB


In [10]:
wl = nltk.WordNetLemmatizer()
ps = nltk.PorterStemmer()

In [11]:
def preprocess(text):
    text = text.lower()
    text = re.sub('[^a-zA-Z\s]','',text)
    stop = stopwords.words('english')
    text = [ps.stem(wl.lemmatize(word)) for word in text.split() if word not in stop]
    return text
    

In [12]:
processed_data = data['Text'].map(preprocess)

In [13]:
processed_data.head()

7819     [day, got, know, would, get, share, dwell, boy...
10433    [text, haircut, rather, haircut, sinc, feel, l...
11421                      [left, feel, littl, dishearten]
4184     [find, interest, supplement, use, without, go,...
21059    [mansel, livid, admit, ran, away, scene, accid...
Name: Text, dtype: object

In [14]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(processed_data)
sequences = tokenizer.texts_to_sequences(processed_data)

word_index = tokenizer.word_index
len(word_index)

12673

In [15]:
features = pad_sequences(sequences, Max_sequence_length)
labels = pd.get_dummies(data['Emotion'])
features.shape, labels.shape

((20000, 50), (20000, 6))

In [17]:
labels.head()

,anger,fear,happy,love,sadness,surprise
7819,0,0,1,0,0,0
10433,0,0,1,0,0,0
11421,0,0,0,0,1,0
4184,0,0,0,0,1,0
21059,1,0,0,0,0,0


In [16]:
x_train, x_test, y_train, y_test = train_test_split(features,labels, test_size = 0.2, random_state = 7)
x_test, x_val, y_test, y_val = train_test_split(features, labels, test_size = 0.5, random_state= 5)

In [19]:
embedding_model = fasttext.load_model(r"D:\Notebook\Projects\Embedding_Models\fasttext_embedding_model.bin")
embedding_matrix = np.zeros((len(word_index)+1, Embedding_dim))
for word,i in word_index.items():
    vector = embedding_model.get_word_vector(word)
    if vector is not None:
        embedding_matrix[i] = vector

embedding_layer = Embedding(len(word_index)+1,Embedding_dim, weights = [embedding_matrix],
                            input_length=Max_sequence_length)(input_sequences)

In [20]:
convs = []
filter_sizes = [3,4,5]

sequence_input = Input(shape=(MAX_SEQUENCE_LENGTH,))
embedded_sequences = embedding_layer(sequence_input)

for fsz in filter_sizes:
    l_conv = Conv1D(128,fsz,activation='relu')(embedded_sequences)
    l_pool = MaxPooling1D(5)(l_conv)
    convs.append(l_pool)   
l_merge = Concatenate()(convs)
l_cov1= Conv1D(filters=128, kernel_size=5, activation='relu')(l_merge)
l_pool1 = MaxPooling1D(5)(l_cov1)
# l_cov2 = Conv1D(filters=128, kernel_size=5, activation='relu')(l_pool1)
# l_pool2 = MaxPooling1D(30)(l_cov2)
l_flat = Flatten()(l_pool1)
l_dense = Dense(128, activation='relu')(l_flat)
preds = Dense(13, activation='softmax')(l_dense)

model = Model(sequence_input, preds)
model.compile(loss='binary_crossentropy',
              optimizer='Nadam',
              metrics=['acc'])

model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 50)]              0         
                                                                 
 embedding (Embedding)       (None, 50, 100)           1267400   
                                                                 
 lstm (LSTM)                 (None, 50, 256)           365568    
                                                                 
 dropout (Dropout)           (None, 50, 256)           0         
                                                                 
 lstm_1 (LSTM)               (None, 50, 128)           197120    
                                                                 
 lstm_2 (LSTM)               (None, 64)                49408     
                                                                 
 dense (Dense)               (None, 6)                 390   

In [21]:
history = model.fit(x_train,y_train, validation_data = (x_val, y_val), epochs = 25, batch_size = 150, verbose=1)

Epoch 1/25
107/107 [==============================] - 69s 574ms/step - loss: 1.4129 - acc: 0.4541 - val_loss: 0.9649 - val_acc: 0.6470
Epoch 2/25
107/107 [==============================] - 52s 482ms/step - loss: 0.6480 - acc: 0.7700 - val_loss: 0.3564 - val_acc: 0.8808
Epoch 3/25
107/107 [==============================] - 50s 464ms/step - loss: 0.2894 - acc: 0.8953 - val_loss: 0.2471 - val_acc: 0.9129
Epoch 4/25
107/107 [==============================] - 51s 476ms/step - loss: 0.1859 - acc: 0.9314 - val_loss: 0.1814 - val_acc: 0.9374
Epoch 5/25
107/107 [==============================] - 58s 541ms/step - loss: 0.1338 - acc: 0.9479 - val_loss: 0.1511 - val_acc: 0.9456
Epoch 6/25
107/107 [==============================] - 56s 524ms/step - loss: 0.1045 - acc: 0.9596 - val_loss: 0.1381 - val_acc: 0.9525
Epoch 7/25
107/107 [==============================] - 52s 486ms/step - loss: 0.0911 - acc: 0.9646 - val_loss: 0.1301 - val_acc: 0.9590
Epoch 8/25
107/107 [==============================] - 5

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline 
# list all data in history
print(history.history.keys())
# summarize history for accuracy
plt.plot(history.history['acc'])
plt.plot(history.history['val_acc'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()
# summarize history for loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [29]:
y_pred = model.predict(x_test, verbose = 1)

313/313 [==============================] - 32s 103ms/step


In [30]:
def fx(array):
    return np.argmax(array)

temp = np.array(y_test)
st_pred = y_pred.tolist()
st_temp = temp.tolist()

arr = np.array(list(map(fx,st_temp)))
ans = np.array(list(map(fx,st_pred)))

In [33]:
f1 = f1_score(arr,ans, average=None)
r1 = recall_score(arr,ans, average=None)
p1 = precision_score(arr,ans, average=None)
acc = accuracy_score(arr,ans)
print('f1',f1)
print('r1',r1)
print('p1',p1)
print('acc',acc)

f1 [0.97179579 0.9692429  0.97841727 0.92708333 0.98175059 0.9398773 ]
r1 [0.97984161 0.97077409 0.97173035 0.91165173 0.98944142 0.93643032]
p1 [0.96388102 0.96771654 0.98519685 0.94304636 0.9741784  0.94334975]
acc 0.9718


In [34]:
f1 = f1_score(arr,ans, average='macro')
p1 = precision_score(arr,ans, average='macro')
r1 = recall_score(arr,ans,average='macro')
print(f1)
print(r1)
print(p1)

0.961361196829492
0.9599782531056892
0.962894820120524


In [35]:
f1 = f1_score(arr,ans, average='micro')
p1 = precision_score(arr,ans, average='micro')
r1 = recall_score(arr,ans,average='micro')
print(f1)
print(r1)
print(p1)

0.9718
0.9718
0.9718


In [36]:
f1 = f1_score(arr,ans, average='weighted')
p1 = precision_score(arr,ans, average='weighted')
r1 = recall_score(arr,ans,average='weighted')
print(f1)
print(r1)
print(p1)

0.9717292678180365
0.9718
0.9717845579805827
